# **Machine Learning Basics**

---



In [4]:
!pip install -q openai chromadb pypdf tiktoken ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [7]:



# ============================================================
# 📦 INSTALLS
# ============================================================

# !pip install -q openai chromadb pypdf tiktoken ipywidgets

# ============================================================
# 📚 IMPORTS
# ============================================================

import uuid
import textwrap

from google.colab import files
from google.colab import userdata

from openai import OpenAI
from pypdf import PdfReader

import chromadb

# ============================================================
# 🔑 OPENAI CLIENT
# ============================================================

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(
    api_key=
)

# ============================================================
# 📄 LOAD PDF
# ============================================================

def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text

# ============================================================
# ✂️ CHUNKER WITH OVERLAP
# ============================================================

def chunk_text(
    text,
    chunk_size=1000,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        # move forward with overlap
        start += chunk_size - overlap

    return chunks

# ============================================================
# 🧠 CREATE EMBEDDING
# ============================================================

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

# ============================================================
# 🗂️ CHROMADB SETUP
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# ============================================================
# 📥 STORE CHUNKS
# ============================================================

def add_chunks_to_chroma(chunks):

    for chunk in chunks:

        embedding = get_embedding(chunk)

        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )

    print(f"Stored {len(chunks)} chunks in ChromaDB.")

# ============================================================
# 🔎 RETRIEVE DOCUMENTS
# ============================================================

def retrieve(query, top_k=3):

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]

    return docs

# ============================================================
# 🤖 ASK QUESTION
# ============================================================

def ask(query):

    docs = retrieve(query)

    context = "\n\n".join(docs)

    prompt = f"""
You are a helpful RAG assistant.

Answer ONLY from the provided context.

If the answer is not in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You answer questions using retrieved documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# ============================================================
# 📂 UPLOAD PDF
# ============================================================

print("Upload your PDF file.\n")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print(f"\nLoaded PDF: {pdf_path}")

# ============================================================
# 🏗️ BUILD RAG PIPELINE
# ============================================================

print("\nLoading PDF...")

text = load_pdf(pdf_path)

print("PDF loaded.")

print("\nChunking document...")

chunks = chunk_text(
    text,
    chunk_size=800,
    overlap=200
)

print(f"Created {len(chunks)} chunks.")

print("\nGenerating embeddings and storing in ChromaDB...")

add_chunks_to_chroma(chunks)

print("\n✅ RAG Pipeline Ready")

# ============================================================
# 💬 INTERACTIVE CHATBOT
# ============================================================

print("\n🤖 Chatbot Ready")
print("Type 'exit' to stop.\n")

while True:

    query = input("You: ")

    if query.lower() == "exit":

        print("\nBot: Goodbye.")
        break

    answer = ask(query)

    print("\nBot:")

    print(textwrap.fill(answer, width=100))

    print("\n" + "=" * 100 + "\n")

Upload your PDF file.



Saving Kajana_Uday_BTech_Provisional_Certificate.pdf to Kajana_Uday_BTech_Provisional_Certificate.pdf

Loaded PDF: Kajana_Uday_BTech_Provisional_Certificate.pdf

Loading PDF...
PDF loaded.

Chunking document...
Created 0 chunks.

Generating embeddings and storing in ChromaDB...
Stored 0 chunks in ChromaDB.

✅ RAG Pipeline Ready

🤖 Chatbot Ready
Type 'exit' to stop.

You: what is the document about

Bot:
The document is about the Code of Conduct, which sets forth the expectation that employees conduct
themselves with integrity at all times. It provides principles to help govern their conduct with
clients, customers, suppliers, and others.


You: whose certficate is this?

Bot:
I could not find the answer in the document.




KeyboardInterrupt: Interrupted by user